# Phase 2: Model Training, Cross-Validation & Serialization

This notebook trains the Automation Risk classifier on the **raw** dataset
(`ai_job_market_insights.csv`), using only the original, verified columns.

`AI_Pressure_Index` and `Salary_Tier` were dropped after testing confirmed
they carry no real signal (see Phase 1 notes) including them would just add
noise, not accuracy.


In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load the raw dataset directly -- no engineered noise features
df = pd.read_csv("ai_job_market_insights.csv")
df.columns = [c.strip() for c in df.columns]

# 2. Curated feature selection
numeric_cols = ['Salary_USD']
cat_cols = [
    'Job_Title', 'Industry', 'Company_Size', 'Location',
    'AI_Adoption_Level', 'Required_Skills', 'Remote_Friendly',
    'Job_Growth_Projection'
]
feature_cols = numeric_cols + cat_cols

X = df[feature_cols]
y = df['Automation_Risk']

# 3. Train-test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [2]:
# 4. Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# 5. Baseline: a model that always guesses the majority class
dummy = Pipeline([('prep', preprocessor), ('clf', DummyClassifier(strategy='most_frequent'))])
dummy.fit(X_train, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test))

In [3]:
# 6. Compare models with 5-fold cross-validation (not a single lucky split)
models = {
    "Majority Baseline": DummyClassifier(strategy='most_frequent'),
    "Logistic Regression (L2)": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
}

print("=== 5-Fold Stratified Cross-Validation Benchmark ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    cv_results[name] = cv_scores.mean()
    print(f"{name:25} | Mean CV: {cv_scores.mean()*100:.2f}% (±{cv_scores.std()*100:.2f}%)")

=== 5-Fold Stratified Cross-Validation Benchmark ===
Majority Baseline         | Mean CV: 34.50% (±0.61%)


Logistic Regression (L2)  | Mean CV: 36.00% (±4.96%)


Random Forest             | Mean CV: 34.75% (±5.02%)


**Read this honestly:** Logistic Regression's cross-validated accuracy is only
a couple of points above the majority baseline, and well within one standard
deviation of it. That means, properly tested, the model's edge over "just guess
Medium every time" is small and not fully reliable — this is the real number to
quote in the pitch, not a single flattering test split.

In [4]:
# 7. Final fit on the full training set, evaluated on the held-out test set
final_pipeline = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

print("=== Holdout Test Performance (Logistic Regression) ===")
print(f"Majority Baseline Test Accuracy : {dummy_acc*100:.2f}%")
print(f"Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%\n")

labels = ['Low', 'Medium', 'High']
cm = confusion_matrix(y_test, y_pred, labels=labels)
print("Confusion Matrix:")
print(pd.DataFrame(cm, index=[f"Actual {l}" for l in labels], columns=[f"Pred {l}" for l in labels]))

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, labels=labels))

=== Holdout Test Performance (Logistic Regression) ===
Majority Baseline Test Accuracy : 35.00%
Logistic Regression Test Accuracy: 44.00%

Confusion Matrix:
               Pred Low  Pred Medium  Pred High
Actual Low           13            5         13
Actual Medium         8           18          9
Actual High          13            8         13

Detailed Classification Report:
              precision    recall  f1-score   support

         Low       0.38      0.42      0.40        31
      Medium       0.58      0.51      0.55        35
        High       0.37      0.38      0.38        34

    accuracy                           0.44       100
   macro avg       0.44      0.44      0.44       100
weighted avg       0.45      0.44      0.44       100



In [5]:
# 8. Save the single, correct pipeline -- nothing downstream overwrites this
joblib.dump(final_pipeline, "automation_risk_model.pkl")
print("Model saved as 'automation_risk_model.pkl'")

Model saved as 'automation_risk_model.pkl'
